# 🚦 turboSH Traffic Analysis (EDA)

**Epic 5, Story 5.1**

The goal of this notebook is to analyze traffic distributions, identify baseline traffic patterns, and visualize traffic spikes. This forms the foundation for anomaly detection models by understanding what "normal traffic" looks like.

## 1. Load the Dataset
We load the raw traffic logs generated by the Traffic Logger middleware.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('ggplot')
plt.rcParams['figure.figsize'] = (12, 5)

# Read the raw JSON Lines logs
df = pd.read_json('../logs/traffic_test.jsonl', lines=True)
df.head()

## 2. Inspect Dataset Structure
Check the shape and columns of the dataset.

In [ ]:
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

## 3. Check for Missing Values

In [ ]:
df.isnull().sum()

## 4. Analyze Traffic Distribution
Which endpoints receive the most traffic?

In [ ]:
df['endpoint'].value_counts()

## 5. Analyze Status Code Distribution
Identify error patterns. Large spikes in 4xx/5xx may indicate scanning or brute-force attacks.

In [ ]:
df['status_code'].value_counts()

## 6. Response Time Distribution
High latency spikes may indicate server overload or attack traffic.

In [ ]:
plt.hist(df['response_time'], bins=50, color='royalblue', edgecolor='white')
plt.title("Response Time Distribution")
plt.xlabel("Latency (ms)")
plt.ylabel("Count")
plt.show()

## 7. Traffic Over Time
Visualizing requests over time helps detect traffic spikes (e.g., DDoS bursts).

In [ ]:
df['timestamp'] = pd.to_datetime(df['timestamp'])

# Resample by a small window (e.g., 5 seconds) since our test data spans 2 minutes
traffic_per_min = df.resample('5S', on='timestamp').size()

traffic_per_min.plot(color='darkorange')
plt.title("Traffic Requests Over Time (5-second windows)")
plt.xlabel("Time")
plt.ylabel("Requests")
plt.show()

## 8. Requests per IP Analysis
IPs making extremely high requests may be bots or attackers.

In [ ]:
requests_per_ip = df.groupby('ip_hash').size()
print("IP Request Summary\n", '-'*20)
print(requests_per_ip.describe())

print("\nTop IPs by request count:\n", '-'*20)
print(requests_per_ip.sort_values(ascending=False).head())

## 9. Endpoint Entropy Analysis
Entropy measures how random endpoint access is. High entropy means an IP accesses many endpoints randomly, which can indicate endpoint scanning attacks.

In [ ]:
# Compute overall endpoint entropy
p = df['endpoint'].value_counts(normalize=True)
entropy = -(p * np.log2(p)).sum()
print(f"Overall Endpoint Entropy: {entropy:.4f}")

## 10. Outlier Detection
Find extreme traffic behavior (e.g., highly suspicious request volumnes or latencies).

In [ ]:
print("Requests with unusually high response time (> 1000ms):")
display(df[df['response_time'] > 1000])

print("\nIPs exceeding baseline request count:")
display(requests_per_ip[requests_per_ip > 10])

## 11. Conclusions
From this EDA, we observe:
- **Normal Request Rate:** Most IPs have a normal request rate around a specific mean.
- **Popular Endpoints:** We were able to identify the most heavily requested endpoints.
- **Latency:** Normal latency is relatively low, while anomalous requests (e.g. cache misses on slow endpoints, or attacks) show significant spikes (> 1000ms).
- **Suspicious behavior:** A minority of IPs exhibit abnormal request volumes, high error rates, or access patterns (high endpoint entropy) indicating a potential scanning or DoS attack.

These insights directly inform our feature engineering (like `error_rate`, `latency_spike`, and `requests_per_ip_60s`) and will act as baselines for the ML models in Epic 6.